In [4]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
# read each csv in the data folder and use seaborn to create analyse / plot stats about categorical data and dates found


In [ ]:

# 1) acq_orders.csv
acq_orders = pd.read_csv('../data/acq_orders.csv')

# print number of records in dataset
print(f"Number of records in acq_orders: {len(acq_orders)}")

display(acq_orders.head(5))
# explore cardinality of taxonomy_business_category_group
print(acq_orders['taxonomy_business_category_group'].value_counts())

# check if any changes in taxonomy_business_category_group per customer_id
customer_category_changes = acq_orders.groupby('customer_id')['taxonomy_business_category_group'].nunique()

#see if multiple groups per user (FALSE)
print(f"Number of customers with category changes: {customer_category_changes[customer_category_changes > 1].count()}")


Number of records in acq_orders: 508694


,customer_id,taxonomy_business_category_group
0,1026819,Weight Loss Group
1,1169512,Weight Loss Group
2,1204351,Weight Loss Group
3,1019780,Weight Loss Group
4,1166196,Weight Loss Group


taxonomy_business_category_group
Hair Loss Group        383273
ED Group                73086
Weight Loss Group       41736
Other Group              7038
Sleep Group              2275
TRT Group                1285
Mental Health Group         1
Name: count, dtype: int64
Number of customers with category changes: 0


In [ ]:
# 2) acq_orders.csv
activity = pd.read_csv('../data/activity.csv')
print(f"Number of records in activity.csv: {len(activity)}")
# print median number of records per customer_id
median_records_per_customer = activity.groupby('customer_id').size().median()
print(f"Median number of records per customer_id: {median_records_per_customer}")

display(activity.head(5))

#check if any to_date <= from_date
activity['from_date'] = pd.to_datetime(activity['from_date'])
activity['to_date'] = pd.to_datetime(activity['to_date'])
invalid_dates = activity[activity['to_date'] < activity['from_date']]
print(f"Number of records with invalid dates: {len(invalid_dates)}")


# check for changes in subscription attributes
# Method 1: Check if count equals unique count
is_unique = activity['subscription_id'].nunique() == len(activity)
print(f"Is subscription_id unique? {is_unique}")

# Method 2: Check for duplicates
has_duplicates = activity['subscription_id'].duplicated().any()
print(f"Has duplicates? {has_duplicates}")

# Method 3: See duplicate counts
duplicate_count = activity['subscription_id'].duplicated().sum()
print(f"Number of duplicates: {duplicate_count}")

# Method 4: Find which subscription_ids are duplicated
if has_duplicates:
    duplicated_subs = activity[activity['subscription_id'].duplicated(keep=False)]
    print("\nDuplicated subscription_ids:")
    print(duplicated_subs.sort_values('subscription_id'))

Number of records in activity.csv: 2176168
Median number of records per customer_id: 3.0


,customer_id,subscription_id,from_date,to_date
0,31922,19621,2019-09-06,2019-10-07
1,3095,36471,2020-05-26,2020-06-19
2,74237,54966,2020-11-02,2021-03-02
3,29966,60229,2020-11-30,2021-03-02
4,64520,46879,2020-12-04,2021-03-02


Number of records with invalid dates: 0
Is subscription_id unique? False
Has duplicates? True
Number of duplicates: 694166

Duplicated subscription_ids:
         customer_id  subscription_id  from_date    to_date
2139655           49               16 2019-01-15 2022-08-23
2159558           49               16 2022-08-25 2024-07-31
1648530           49               16 2024-08-01 2024-08-06
969482           326               73 2023-11-16 2023-11-20
2007932          326               73 2021-05-24 2023-11-10
...              ...              ...        ...        ...
1615636       203216          1761704 2024-08-15 2024-08-15
2174783       427167          1762204 2024-08-16 2024-08-16
1654831       427167          1762204 2024-08-15 2024-08-16
1220259       427167          1762205 2024-08-16 2024-08-16
2098949       427167          1762205 2024-08-15 2024-08-16

[1010500 rows x 4 columns]


In [21]:
# print number of records in customers.csv
customers = pd.read_csv('../data/customers.csv')
display(customers.head(5))

# check cardinality of customer_country
print(customers['customer_country'].value_counts())

customer_country_counts = customers.groupby('customer_id')['customer_country'].nunique()
if (customer_country_counts > 1).any():
    print("There are customers with multiple countries.")


,customer_id,customer_country
0,201441,United Kingdom
1,122538,United Kingdom
2,38482,United Kingdom
3,7792,United Kingdom
4,289880,United Kingdom


customer_country
Brazil            296693
United Kingdom    236155
Name: count, dtype: int64


In [2]:
# use duck db to read  ../data/activity and query for records with customer_id = 948
import duckdb
con = duckdb.connect()
query = """
SELECT  * from '../data/activity.csv'
WHERE customer_id = 948
Order by from_date
""" 
result = con.execute(query).fetchdf()
display(result)
con.close()

,customer_id,subscription_id,from_date,to_date
0,948,638,2019-02-18,2019-03-18
1,948,18788,2019-08-20,2019-09-19
2,948,18788,2019-09-21,2019-10-23
3,948,22647,2019-10-24,2019-11-19
4,948,34269,2020-05-02,2020-06-01
5,948,34270,2020-05-02,2020-06-01
6,948,36961,2020-06-01,2020-07-01
7,948,36961,2020-07-03,2020-07-31
8,948,36961,2020-08-05,2020-08-27
9,948,492936,2022-08-17,2023-01-23
